In [ ]:
#import libraries
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn

In [ ]:
#variables
batch_size = 64
img_size = 28
patch_size = 7
num_channels = 1
num_patches = (img_size // patch_size) ** 2
num_heads = 1
embed_dim = 16
mlp_dim = 16
transformer_units = 1

In [ ]:
#Python image library (PIL) to tensor
transform = transforms.Compose(
    [transforms.ToTensor()])

In [ ]:
#load dataset
trainset = torchvision.datasets.MNIST(root='../data', train=True,
                                        download=True, transform=transform)
valset = torchvision.datasets.MNIST(root='../data', train=False,
                                        download=True, transform=transform)

In [ ]:
#create train and val batches
train_data = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
val_data = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                          shuffle=False)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        return x

In [ ]:
class TransformerArchitecture(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.multi_layer_perceptron = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )

    def forward(self, x):
        residual_1 = x
        attention_output = self.self_attention(self.layer_norm_1(x),self.layer_norm_1(x),self.layer_norm_1(x))[0]
        x = attention_output + residual_1
        residual_2 = x
        mlp_output = self.multi_layer_perceptron(self.layer_norm_2(x))
        x = mlp_output + residual_2
        return x


In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, (img_size // patch_size) ** 2 + 1, embed_dim))
        self.transformer_layers = nn.Sequential(*[TransformerArchitecture() for _ in range(transformer_units)])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 10)
        )

    def forward(self,x):
        x = self.patch_embedding(x)
        B = x.size(0)

        cls_tokens = self.cls_token.expand(B , -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.transformer_layers(x)
        x = x[:,0]
        x = self.mlp_head(x)
        return x

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct_epoch = 0
    total_epoch = 0
    print(f"\nEpoch {epoch+1}")

    for batch_idx, (images, labels) in enumerate(train_data):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        preds = outputs.argmax(dim=1)

        correct = (preds == labels).sum().item()
        accuracy = 100.0 * correct / labels.size(0)

        correct_epoch += correct
        total_epoch += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx+1:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")

    epoch_acc = 100.0 * correct_epoch / total_epoch
    print(f"==> Epoch {epoch+1} Summary: Total Loss = {total_loss:.4f}, Accuracy = {epoch_acc:.2f}%")

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_data:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = 100.0 * correct / total
print(f"\n==> Val Accuracy: {test_acc:.2f}%")


In [ ]:
model.eval()
images, labels = next(iter(val_data))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = outputs.argmax(dim=1)

images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

# Plot 10 test images with predictions
plt.figure(figsize=(12, 4))
for i in range(20):
    plt.subplot(4, 5, i+1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f"Pred: {preds[i].item()}\nTrue: {labels[i].item()}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## Visualizing the architecture

Data flow for this config (B = batch):

| Stage | Shape |
|---|---|
| Input | `[B, 1, 28, 28]` |
| PatchEmbedding (Conv2d 7x7, stride 7) | `[B, 16, 4, 4]` |
| Flatten + transpose (16 patches x 16 dims) | `[B, 16, 16]` |
| Prepend CLS token | `[B, 17, 16]` |
| + pos_embed | `[B, 17, 16]` |
| Transformer block x1: `x + MHA(LN1(x))`, `x + MLP(LN2(x))` | `[B, 17, 16]` |
| CLS output `x[:, 0]` | `[B, 16]` |
| LayerNorm + Linear(16, 10) | `[B, 10]` |


In [ ]:
# Module tree (built-in, no install)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# Layer-by-layer shapes and parameter counts
# Requires: uv pip install torchinfo
from torchinfo import summary

summary(model, input_size=(1, num_channels, img_size, img_size),
        col_names=("input_size", "output_size", "num_params"),
        device=device)


In [ ]:
# Computation graph
# Requires: uv pip install torchview  AND  brew install graphviz
from torchview import draw_graph

draw_graph(VisionTransformer(), input_size=(1, num_channels, img_size, img_size),
           expand_nested=True, depth=3, device="cpu").visual_graph


## What the model learned

Attention maps, positional embedding similarity and patch embedding filters of the trained model.


In [ ]:
# CLS-token attention over the 16 patches, overlaid on the input digit
import torch.nn.functional as F

attn = {}
def save_attn(module, inp, out):
    # out = (attn_output, attn_weights); weights averaged over heads: (B, 17, 17)
    attn["w"] = out[1].detach().cpu()

hook = model.transformer_layers[0].self_attention.register_forward_hook(save_attn)

model.eval()
images, labels = next(iter(val_data))
with torch.no_grad():
    preds = model(images.to(device)).argmax(1).cpu()
hook.remove()

grid = img_size // patch_size
cls_attn = attn["w"][:, 0, 1:].reshape(-1, 1, grid, grid)
cls_attn = F.interpolate(cls_attn, size=(img_size, img_size), mode="nearest")

plt.figure(figsize=(14, 4))
for i in range(8):
    plt.subplot(2, 8, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"P{preds[i].item()} T{labels[i].item()}")
    plt.axis("off")
    plt.subplot(2, 8, i + 9)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.imshow(cls_attn[i, 0], cmap="jet", alpha=0.5)
    plt.axis("off")
plt.suptitle("Top: input (P=pred, T=true). Bottom: CLS attention")
plt.tight_layout()
plt.show()


In [ ]:
# Positional embedding cosine similarity: each small grid = one patch position vs all others
import torch.nn.functional as F

grid = img_size // patch_size
pe = F.normalize(model.pos_embed[0, 1:].detach().cpu(), dim=-1)  # (num_patches, embed_dim)
sim = pe @ pe.T

fig, axes = plt.subplots(grid, grid, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(sim[i].reshape(grid, grid), cmap="viridis", vmin=-1, vmax=1)
    ax.axis("off")
fig.suptitle("Positional embedding similarity")
plt.show()


In [ ]:
# Learned patch embedding filters (Conv2d kernels)
w = model.patch_embedding.patch_embed.weight.detach().cpu()  # (embed_dim, 1, patch_size, patch_size)

fig, axes = plt.subplots(2, embed_dim // 2, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(w[i, 0], cmap="gray")
    ax.axis("off")
fig.suptitle("Patch embedding filters")
plt.show()


In [ ]:
# Get one batch from the validation data
images, labels = next(iter(val_data))

# Print the shape of the images tensor
print("Shape of images in a batch:", images.shape)
patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
embedded_data = patch_embed(images)
embedded_data = embedded_data.flatten(2)
embedded_data = embedded_data.transpose(1,2)
print("Shape of embedded data:", embedded_data.shape)
print(torch.randn(1,1,embed_dim).shape)

# You can also print the shape of the labels tensor if you'd like
# print("Shape of labels in a batch:", labels.shape)

### Visualize

In [ ]:
from torchview import draw_graph
draw_graph(VisionTransformer(), input_size=(1, 1, 28, 28),
           expand_nested=True, depth=3, device="cpu").visual_graph